### Import Movie Lens Data for Recommendation System

##### This combines movielens, letterboxd, and tmdb data

In [1]:
import pandas as pd

In [2]:
ml_ratings = pd.read_csv("../data/movielens/ratings.csv")
ml_movies = pd.read_csv("../data/movielens/movies.csv")
ml_links = pd.read_csv("../data/movielens/links.csv")


In [3]:
user_movies = pd.read_csv("../data/enriched_movies.csv")
user_movies.head()

,Name,Year,Rating,tmdb_id,genres,director,cast
0,The Usual Suspects,1995,5.0,629.0,"['Drama', 'Crime', 'Thriller']",Bryan Singer,"['Stephen Baldwin', 'Gabriel Byrne', 'Benicio ..."
1,Good Will Hunting,1997,5.0,489.0,['Drama'],Gus Van Sant,"['Matt Damon', 'Robin Williams', 'Ben Affleck'..."
2,Scream,1996,5.0,4232.0,"['Crime', 'Horror', 'Mystery']",Wes Craven,"['Neve Campbell', 'Skeet Ulrich', 'Rose McGowa..."
3,A Few Good Men,1992,5.0,881.0,['Drama'],Rob Reiner,"['Tom Cruise', 'Jack Nicholson', 'Demi Moore',..."
4,No Country for Old Men,2007,5.0,6977.0,"['Crime', 'Thriller', 'Western']",Joel Coen,"['Javier Bardem', 'Tommy Lee Jones', 'Josh Bro..."


In [19]:
my_ratings["Rating"].value_counts().sort_index()

Rating
1.5     2
2.0     2
2.5    15
3.0    55
3.5    57
4.0    68
4.5    46
5.0    19
Name: count, dtype: int64

In [50]:
letterboxd_ml_merged.head()

,Name,Year,Rating,tmdb_id,genres_x,director,cast,movieId,title,genres_y,imdbId,tmdbId
0,The Usual Suspects,1995,5.0,629.0,"['Drama', 'Crime', 'Thriller']",Bryan Singer,"['Stephen Baldwin', 'Gabriel Byrne', 'Benicio ...",50.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,114814.0,629.0
1,Good Will Hunting,1997,5.0,489.0,['Drama'],Gus Van Sant,"['Matt Damon', 'Robin Williams', 'Ben Affleck'...",1704.0,Good Will Hunting (1997),Drama|Romance,119217.0,489.0
2,Scream,1996,5.0,4232.0,"['Crime', 'Horror', 'Mystery']",Wes Craven,"['Neve Campbell', 'Skeet Ulrich', 'Rose McGowa...",1407.0,Scream (1996),Comedy|Horror|Mystery|Thriller,117571.0,4232.0
3,A Few Good Men,1992,5.0,881.0,['Drama'],Rob Reiner,"['Tom Cruise', 'Jack Nicholson', 'Demi Moore',...",2268.0,"Few Good Men, A (1992)",Crime|Drama|Thriller,104257.0,881.0
4,No Country for Old Men,2007,5.0,6977.0,"['Crime', 'Thriller', 'Western']",Joel Coen,"['Javier Bardem', 'Tommy Lee Jones', 'Josh Bro...",55820.0,No Country for Old Men (2007),Crime|Drama,477348.0,6977.0


In [4]:
print(ml_ratings.shape)
print(ml_movies.shape)
print(ml_links.shape)

(100836, 4)
(9742, 3)
(9742, 3)


In [31]:
ml_ratings_tmdb = ml_ratings.merge(ml_links, on="movieId", how="left")
ml_ratings_tmdb.head()

,userId,movieId,rating,timestamp,imdbId,tmdbId
0,1,1,4.0,964982703,114709,862.0
1,1,3,4.0,964981247,113228,15602.0
2,1,6,4.0,964982224,113277,949.0
3,1,47,5.0,964983815,114369,807.0
4,1,50,5.0,964982931,114814,629.0


In [5]:
ml_links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [6]:
# Add the tmdbId column to the ml_movies dataframe by merging it with the ml_links dataframe on the movieId column
ml_join_data = pd.merge(ml_movies, ml_links, on="movieId", how="left")
ml_join_data.head()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


In [106]:
ml_join_data.to_csv("../data/movielens/ml_movies_with_tmdbId.csv", index=False)

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/scottsieger/letterboxd-recommender/.venv/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 566, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/scottsieger/letterboxd-recommender/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/Users/scottsieger/letterboxd-recommender/.venv/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/scottsieger/letterboxd-recommender/.venv/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list


In [7]:
print(ml_join_data.dtypes)
print(user_movies.dtypes)

# Both tmdb IDs are float64, so we can merge on them

movieId      int64
title          str
genres         str
imdbId       int64
tmdbId     float64
dtype: object
Name            str
Year          int64
Rating      float64
tmdb_id     float64
genres          str
director        str
cast            str
dtype: object


In [70]:
# Merge the user's letterboxd enriched movie data with the movielens data on the tmdb_id column
letterboxd_ml_merged = pd.merge(user_movies, ml_join_data, left_on="tmdb_id", right_on="tmdbId", how="left")
letterboxd_ml_merged.head()

,Name,Year,Rating,tmdb_id,genres_x,director,cast,movieId,title,genres_y,imdbId,tmdbId
0,The Usual Suspects,1995,5.0,629.0,"['Drama', 'Crime', 'Thriller']",Bryan Singer,"['Stephen Baldwin', 'Gabriel Byrne', 'Benicio ...",50.0,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,114814.0,629.0
1,Good Will Hunting,1997,5.0,489.0,['Drama'],Gus Van Sant,"['Matt Damon', 'Robin Williams', 'Ben Affleck'...",1704.0,Good Will Hunting (1997),Drama|Romance,119217.0,489.0
2,Scream,1996,5.0,4232.0,"['Crime', 'Horror', 'Mystery']",Wes Craven,"['Neve Campbell', 'Skeet Ulrich', 'Rose McGowa...",1407.0,Scream (1996),Comedy|Horror|Mystery|Thriller,117571.0,4232.0
3,A Few Good Men,1992,5.0,881.0,['Drama'],Rob Reiner,"['Tom Cruise', 'Jack Nicholson', 'Demi Moore',...",2268.0,"Few Good Men, A (1992)",Crime|Drama|Thriller,104257.0,881.0
4,No Country for Old Men,2007,5.0,6977.0,"['Crime', 'Thriller', 'Western']",Joel Coen,"['Javier Bardem', 'Tommy Lee Jones', 'Josh Bro...",55820.0,No Country for Old Men (2007),Crime|Drama,477348.0,6977.0


In [71]:
# How many movies in my letterboxd history are also in the movielens dataset?
coverage = (
    letterboxd_ml_merged["movieId"]
    .notna()
    .mean()
)

print(
    f"MovieLens coverage of my Letterboxd history: "
    f"{coverage:.1%}"
)

MovieLens coverage of my Letterboxd history: 55.7%


In [72]:
# Add a unique User ID to the letterboxd_ml_merged dataframe for the user
MY_USER_ID = ml_ratings["userId"].max() + 1
print(f"Assigning userId {MY_USER_ID} to the user")

Assigning userId 611 to the user


In [73]:
letterboxd_ml_merged.isna().sum()

Name          0
Year          0
Rating        0
tmdb_id       8
genres_x      8
director      9
cast          8
movieId     117
title       117
genres_y    117
imdbId      117
tmdbId      125
dtype: int64

In [99]:
combined_ratings[combined_ratings["tmdbId"].isna()]

,userId,tmdbId,rating
282,3,<NA>,5.0
14095,90,<NA>,4.0
16539,105,<NA>,5.0
23344,160,<NA>,3.5
30090,210,<NA>,4.0
31391,217,<NA>,3.0
42665,288,<NA>,2.0
44588,298,<NA>,0.5
69291,448,<NA>,0.5
72810,469,<NA>,3.0


In [ ]:
# Filter out Stranger Things rating as it does not have a tmdbId and is not a movie
my_ratings = letterboxd_ml_merged.dropna(subset=["tmdb_id"])[["tmdb_id", "Rating"]].copy()

In [76]:
my_ratings.isna().sum()

tmdb_id    0
Rating     0
dtype: int64

In [77]:
# Match Movie Lens formatting
my_ratings = my_ratings.rename(
    columns={
        "Rating": "rating",
        "tmdb_id": "tmdbId"
    }
)

my_ratings["userId"] = MY_USER_ID

my_ratings["tmdbId"] = (
    my_ratings["tmdbId"]
    .astype("Int64")
    .where(my_ratings["tmdbId"] != -1, None)
)

In [78]:
my_ratings.head(10)

,tmdbId,rating,userId
0,629,5.0,611
1,489,5.0,611
2,4232,5.0,611
3,881,5.0,611
4,6977,5.0,611
5,28,5.0,611
6,500,5.0,611
7,120467,5.0,611
8,1124,4.5,611
9,16869,4.0,611


In [79]:
# Combine the user's ratings with the MovieLens ratings
combined_ratings = pd.concat([ml_ratings_tmdb[["userId", "tmdbId", "rating"]], my_ratings[["userId", "tmdbId", "rating"]]], ignore_index=True)
combined_ratings.head()

,userId,tmdbId,rating
0,1,862.0,4.0
1,1,15602.0,4.0
2,1,949.0,4.0
3,1,807.0,5.0
4,1,629.0,5.0


In [100]:
# Filter out any rows with missing tmdbId values
combined_ratings = combined_ratings.dropna(subset=["tmdbId"])

In [80]:
my_ratings[my_ratings["userId"]==611]["rating"].value_counts().sort_index()

rating
1.5     2
2.0     2
2.5    15
3.0    47
3.5    57
4.0    68
4.5    46
5.0    19
Name: count, dtype: int64

In [101]:
combined_ratings.to_csv("../data/user_movielens_ratings.csv", index=False)

In [102]:
# Create a sparse matrix of the combined ratings data
from scipy.sparse import csr_matrix

user_codes = combined_ratings["userId"].astype("category")
movie_codes = combined_ratings["tmdbId"].astype("category")

user_index = user_codes.cat.codes
movie_index = movie_codes.cat.codes

In [103]:
# Create sparse matrix of ratings with shape (num_users, num_movies)
rating_matrix = csr_matrix(
    (combined_ratings["rating"], (user_index, movie_index)),
    shape=(user_codes.cat.categories.size, movie_codes.cat.categories.size))

In [104]:
rating_matrix.shape

(611, 9830)

In [105]:
rating_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 101076 stored elements and shape (611, 9830)>

##### Popularity Based Recommendation System

In [20]:
# Get movies with highest average ratings
combined_ratings_w_name = combined_ratings.merge(ml_movies[["movieId", "title"]], on="movieId")

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"


In [31]:
# Get movies with more than 10 ratings
movie_thresh = combined_ratings_w_name.groupby("movieId")["userId"].count().sort_values(ascending=False)
movie_thresh = movie_thresh[movie_thresh >= 10]

In [32]:
movie_thresh.shape

(2272,)

In [28]:
movie_thresh.head()

movieId
356     329
318     318
296     308
593     280
2571    278
Name: userId, dtype: int64

In [34]:
combined_ratings_final = combined_ratings_w_name[combined_ratings_w_name["movieId"].isin(movie_thresh.index)]
combined_ratings_final.head()

,userId,movieId,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"


In [36]:
# Get movies with most number of watchers
movie_freq = combined_ratings_final.groupby("title")["userId"].count().sort_values(ascending=False)

In [37]:
# Get highest rated movies with at least 10 ratings
highest_rated_movies = combined_ratings_final.groupby("title").mean().sort_values(("rating"), ascending=False)

In [ ]:
popular_movies = pd.DataFrame({"avg_rating": highest_rated_movies["rating"], "num_ratings": movie_freq})

,avg_rating,num_ratings
title,,
"'burbs, The (1989)",3.176471,17
(500) Days of Summer (2009),3.666667,42
10 Cloverfield Lane (2016),3.678571,14
10 Things I Hate About You (1999),3.527778,54
"10,000 BC (2008)",2.705882,17
101 Dalmatians (1996),3.074468,47
101 Dalmatians (One Hundred and One Dalmatians) (1961),3.431818,44
12 Angry Men (1957),4.155172,58
12 Years a Slave (2013),3.625000,16


In [39]:
popular_movies.sort_values("avg_rating", ascending=False).head(10)

,avg_rating,num_ratings
title,,
Secrets & Lies (1996),4.590909,11
Guess Who's Coming to Dinner (1967),4.545455,11
Paths of Glory (1957),4.541667,12
"Streetcar Named Desire, A (1951)",4.475000,20
"Celebration, The (Festen) (1998)",4.458333,12
Ran (1985),4.433333,15
"Shawshank Redemption, The (1994)",4.429245,318
His Girl Friday (1940),4.392857,14
All Quiet on the Western Front (1930),4.350000,10


In [40]:
# Create a function to get recommendations based on rating with a minimum number of ratings
def top_movies(min_ratings=10, top_n=10):
    popular_movies_filtered = popular_movies[popular_movies["num_ratings"] >= min_ratings]
    return popular_movies_filtered.sort_values("avg_rating", ascending=False).head(top_n)

In [ ]:
# Get movie recommendations with at least 100 ratings and return the top 10
top_movies(100, 10)

,avg_rating,num_ratings
title,,
"Shawshank Redemption, The (1994)",4.429245,318
"Godfather, The (1972)",4.289062,192
Fight Club (1999),4.271689,219
"Godfather: Part II, The (1974)",4.259690,129
Goodfellas (1990),4.251969,127
"Departed, The (2006)",4.250000,108
"Usual Suspects, The (1995)",4.241463,205
Casablanca (1942),4.240000,100
"Dark Knight, The (2008)",4.236667,150
